In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision"
setup_plotting(figure_dir, display_dpi=200, save_dpi=300)

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/08.2-kidney_tcr_clonal_clusters_peri_glom_annot.h5ad"
adata = sc.read_h5ad(path)

# remove control samples
adata = adata[adata.obs["condition"] == "ANCA"].copy()
sample_mapping = {k: f"S{i}" for i, k in enumerate(adata.obs["sample"].unique())}
adata.obs["sample_short"] = adata.obs["sample"].map(sample_mapping)
adata

In [ ]:
adata.obs.columns

In [ ]:
adata.obs["sample"]

In [ ]:
import json

with open("data/xenium/sample_to_patient_mapping.json") as f:
    sample_mapping = json.load(f)
sample_mapping

In [ ]:
anca_mapping = {
    "XETG00088__0029040__Region_4__20240719__095642": "ANCA 0",
    "XETG00088__0029040__Region_5__20240719__095642": "ANCA 1",
    "XETG00088__0029040__Region_6__20240719__095642": "ANCA 2",
    "XETG00088__0029040__Region_7__20240719__095642": "ANCA 3",
    "XETG00088__0029041__Region_3__20240719__095642": "ANCA 4",
    "XETG00088__0029041__Region_4__20240719__095642": "ANCA 5",
    "XETG00088__0029041__Region_5__20240719__095642": "ANCA 6",
    "XETG00088__0029041__Region_6__20240719__095642": "ANCA 7",
    "XETG00088__0029041__Region_7__20240719__095642": "ANCA 8",
    "XETG00088__0029041__Region_8__20240719__095642": "ANCA 9",
}

In [ ]:
patient_data = pd.read_excel("data/xenium/patient.xlsx")
patient_data = patient_data[patient_data["#"].notna()].copy().set_index("#")

In [ ]:
adata.obs["ANCA_status"] = (
    adata.obs["sample"].map(sample_mapping).str[1::].map(patient_data["ANCA"].to_dict())
)

In [ ]:
adata.obs["anca_sample"] = adata.obs["sample"].map(anca_mapping)

In [ ]:
adata.obs["sample_short_anca"] = (
    adata.obs["anca_sample"].astype(str)
    + "\n("
    + adata.obs["ANCA_status"].astype(str)
    + ")"
)

In [ ]:
sample_key = "sample_short_anca"
data = (
    adata.obs.loc[adata.obs["avbv_cluster_filtered"].notna()]
    .groupby(sample_key)["avbv_cluster_filtered"]
    .nunique()
    .sort_values(ascending=False)
)

# tidy for seaborn
df = data.rename("n_unique_clusters").reset_index()
order = df.sort_values("n_unique_clusters", ascending=False)[sample_key]

display(df)

# plot
# sns.set_theme(style="whitegrid", context="talk")
fig, ax = plt.subplots(figsize=(max(6, 0.5 * len(df)), 4))

sns.barplot(
    data=df,
    x=sample_key,
    y="n_unique_clusters",
    order=order,
    ax=ax,
    errorbar=None,
)

ax.set(
    title="Unique avbv clusters per sample",
)
ax.set_ylabel("number of unique clusters", fontsize=10)
ax.set_xlabel("sample", fontsize=10)

ax.tick_params(axis="x", rotation=0, labelsize=8)
ax.margins(x=0.01)
sns.despine(ax=ax, left=False, bottom=False)
fig.tight_layout()
plt.show()